# 04 · Memory 持久化：短期记忆与长期记忆

这一节只验证两个不同的作用域：

| 类型 | 作用域 | LangGraph 机制 | 本页证据 |
|---|---|---|---|
| **短期记忆** | 同一个会话线程 | State + checkpointer + 同一 `thread_id` | 同 thread 能续聊；新 thread 不带上一轮消息 |
| **长期记忆** | 同一用户、跨多个线程 | Store + `user_id` namespace | 同一 user 换 thread 仍读到偏好；不同 user 不串用 |

> **边界：**本 notebook 使用 `InMemorySaver` 与 `InMemoryStore`，适合单进程课堂演示。它们能证明“按 thread / user 分层”的语义，但进程退出后会清空。跨重启耐久见 `04-memory-postgres-demo.ipynb`（本地 Docker + `PostgresSaver`）。

## 怎么跑

1. 打开 `04-memory.ipynb`，Kernel 选 **OOCL 2026 AI Agent**。
2. 确认训练仓根目录 `.env` 已配置 `LLM_BASE_URL`、`LLM_MODEL` 和所需的 `LLM_API_KEY`。
3. 执行 **Run → Run All Cells**。
4. 依次看到 `04 short-term memory ok`、`04 long-term memory ok` 与 `04 memory demo ok` 即通关。


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Annotated, TypedDict

from dotenv import load_dotenv

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / ".env.example").exists():
        ROOT = candidate
        break

os.chdir(ROOT)
load_dotenv(ROOT / ".env")

print("cwd =", ROOT)
print("LLM_BASE_URL =", os.getenv("LLM_BASE_URL"))
print("LLM_MODEL =", os.getenv("LLM_MODEL"))
print("short-term path = live LLM")
print("long-term path = deterministic Store demo")


## 1. 短期记忆：checkpoint 保存 thread-scoped State

短期记忆属于图的运行 State。本例把 `messages` 写入 checkpoint：

- 相同 `thread_id` 再次 `invoke`：加载该 thread 的历史消息。
- 更换 `thread_id`：创建新的 thread，不应看到旧会话。
- `graph.get_state(config)`：可以直接检查当前 thread 的 checkpoint，而不是只看模型最终回答。


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

SYSTEM = (
    "你只用于课堂演示会话记忆，不提供实际港口、危险品或订舱法规结论；每次最多回答两句话。"
    "若历史中有港口与货种，只复述当前语境，并说明真实要求应查询受控规则源。"
    "若用户用『那…呢』追问且历史中有货种或主题，必须沿用该语境。"
    "若历史中没有可供指代的内容，必须明确说明缺少上一轮语境，并要求用户补充港口与货种。"
)


class ChatState(TypedDict):
    messages: Annotated[list, add_messages]


def make_llm(temperature: float = 0):
    """从 .env 构造本练习的真实模型。"""
    from langchain_openai import ChatOpenAI

    base_url = (os.getenv("LLM_BASE_URL") or "").strip()
    model = (os.getenv("LLM_MODEL") or "").strip()
    if not base_url or not model:
        raise ValueError("请在 .env 填写 LLM_BASE_URL 和 LLM_MODEL")
    return ChatOpenAI(
        model=model,
        api_key=os.getenv("LLM_API_KEY") or "not-required",
        base_url=base_url,
        temperature=temperature,
    )


llm = make_llm()


def chat_node(state: ChatState) -> dict:
    messages = list(state.get("messages") or [])
    if not messages or not isinstance(messages[0], SystemMessage):
        messages = [SystemMessage(content=SYSTEM), *messages]
    return {"messages": [llm.invoke(messages)]}


short_builder = StateGraph(ChatState)
short_builder.add_node("chat", chat_node)
short_builder.add_edge(START, "chat")
short_builder.add_edge("chat", END)

short_checkpointer = InMemorySaver()
short_app = short_builder.compile(checkpointer=short_checkpointer)

print("short-term checkpointer =", type(short_checkpointer).__name__)


## 2. 短期记忆对照：同 thread 续聊，新 thread 失忆

先在 `booking-demo-1` 中建立“上海港 + 锂电池”语境，再用同一个 thread 追问“那洛杉矶港呢？”。随后换一个全新的 thread，发送相同追问。


In [ ]:
# TODO(training): 完成“同 thread 续聊、新 thread 失忆”的短期记忆对照实验。
# 1. 使用固定 thread_id 连续调用 short_app.invoke 两轮，验证第二轮能继承第一轮上下文。
# 2. 使用新的 thread_id 提交相同追问，验证新 thread 不会继承旧上下文。
# 3. 使用 short_app.get_state(...) 读取 checkpoint，并检查消息数量与 checkpoint_id。
# 4. 添加断言，分别证明同 thread 有记忆、新 thread 无记忆。


## 3. 长期记忆：Store 按 user namespace 跨 thread 读取

长期记忆不依赖某个会话 thread。下面把“回复语言、回答风格”保存到用户级 Store：

- 同一个 `user_id` 换新 `thread_id`：仍能读取偏好。
- 不同 `user_id`：必须得到空偏好，不能串用。
- 这里只保存用户偏好；业务规则、港口限制和合规结论仍来自受控知识源或工具，不能被长期记忆覆盖。


In [ ]:
from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore


class UserContext(TypedDict):
    user_id: str


class LongTermState(TypedDict, total=False):
    preference_update: dict[str, str]
    loaded_preference: dict[str, str]
    response: str


USER_MEMORY_NS = ("booking_assistant", "user_preferences")


def persist_user_preference(
    state: LongTermState,
    runtime: Runtime[UserContext],
) -> dict:
    update = dict(state.get("preference_update") or {})
    if update:
        runtime.store.put(USER_MEMORY_NS, runtime.context["user_id"], update)
    return {}


def load_user_preference(
    state: LongTermState,
    runtime: Runtime[UserContext],
) -> dict:
    item = runtime.store.get(USER_MEMORY_NS, runtime.context["user_id"])
    return {"loaded_preference": dict(item.value) if item else {}}


def render_preference(state: LongTermState) -> dict:
    preference = state.get("loaded_preference") or {}
    if not preference:
        return {"response": "未找到该用户的长期偏好；使用系统默认输出。"}
    return {
        "response": (
            f"读取长期偏好：language={preference.get('language')}; "
            f"answer_style={preference.get('answer_style')}"
        )
    }


long_builder = StateGraph(LongTermState, context_schema=UserContext)
long_builder.add_node("persist_user_preference", persist_user_preference)
long_builder.add_node("load_user_preference", load_user_preference)
long_builder.add_node("render_preference", render_preference)
long_builder.add_edge(START, "persist_user_preference")
long_builder.add_edge("persist_user_preference", "load_user_preference")
long_builder.add_edge("load_user_preference", "render_preference")
long_builder.add_edge("render_preference", END)

long_store = InMemoryStore()
long_checkpointer = InMemorySaver()
long_app = long_builder.compile(
    checkpointer=long_checkpointer,
    store=long_store,
)

print("long-term store =", type(long_store).__name__)


## 4. 长期记忆对照：同 user 跨 thread 保留，不同 user 隔离


In [ ]:
USER_A = {"user_id": "customer-A"}
USER_B = {"user_id": "customer-B"}

# thread A-1 写入用户 A 的长期偏好
# TODO(training): 调用 long_app.invoke，在线程 A-1 中写入 USER_A 的长期偏好，并将结果保存为 written。
print("写入后:", written["response"])

# 新 thread A-2，同一个 user_id：仍能读取 Store
# TODO(training): 调用 long_app.invoke，使用新 thread A-2 和 USER_A，将结果保存为 same_user_new_thread。
print("同 user / 新 thread:", same_user_new_thread["response"])

# 新 thread B-1，不同 user_id：不得读取用户 A 的偏好
# TODO(training): 调用 long_app.invoke，使用新 thread B-1 和 USER_B，将结果保存为 different_user。
print("不同 user:", different_user["response"])

stored_item = long_store.get(USER_MEMORY_NS, USER_A["user_id"])
print("Store item:", stored_item.value)

assert same_user_new_thread["loaded_preference"]["language"] == "中文"
assert same_user_new_thread["loaded_preference"]["answer_style"] == "先列证据缺口"
assert different_user["loaded_preference"] == {}
assert stored_item.value["language"] == "中文"
print("04 long-term memory ok")


## 5. 通关检查与生产边界

| 检查 | 合格证据 |
|---|---|
| 短期记忆 | 同 `thread_id` 的 checkpoint 含多轮 messages；新 thread 不带旧消息 |
| 长期记忆 | 同 `user_id` 换 thread 仍能读取 Store；不同 user 为空 |
| 权威性 | 长期记忆只保存偏好或经批准的摘要，不保存为“当前有效规则” |
| 跨重启耐久 | 课堂 In-memory 后端不满足；生产应选择数据库支持的 checkpointer 与 Store，并定义权限、保留期、更新和删除策略 |

API 与具体后端以课程锁定依赖和当前 LangGraph 官方文档为准。


In [ ]:
assert type(short_checkpointer).__name__ == "InMemorySaver"
assert type(long_store).__name__ == "InMemoryStore"
assert "checkpoint_id" in snapshot.config["configurable"]
assert long_store.get(USER_MEMORY_NS, "customer-B") is None

print("04 memory demo ok")


<!-- codex:p0:04 -->
## P0 进阶 · Checkpoint 历史、Replay 与 Fork

`get_state()` 只查看当前 checkpoint；`get_state_history()` 可以查看同一 thread 的完整执行历史。

- **Replay：**从旧 checkpoint 继续，旧节点结果保留，后续节点重新执行。
- **Fork：**先用 `update_state()` 基于旧 checkpoint 创建新分支，再从新分支继续；原历史不会被覆盖。

为避免重复调用模型，下面使用确定性的订舱预审图演示 time travel。

In [ ]:
class TravelState(TypedDict, total=False):
    port: str
    cargo: str
    verdict: str


def deterministic_review(state: TravelState) -> dict:
    return {"verdict": f"reviewed:{state['port']}:{state['cargo']}"}


travel_builder = StateGraph(TravelState)
travel_builder.add_node("review", deterministic_review)
travel_builder.add_edge(START, "review")
travel_builder.add_edge("review", END)
travel_app = travel_builder.compile(checkpointer=InMemorySaver())

travel_config = {"configurable": {"thread_id": "time-travel-booking-1"}}
original = travel_app.invoke(
    {"port": "Shanghai", "cargo": "lithium-battery"},
    travel_config,
)

history = list(travel_app.get_state_history(travel_config))
for item in history:
    print(
        "step =", item.metadata.get("step"),
        "next =", item.next,
        "checkpoint_id =", item.config["configurable"].get("checkpoint_id"),
    )

before_review = next(item for item in history if item.next == ("review",))
replayed = travel_app.invoke(None, before_review.config)

fork_config = travel_app.update_state(
    before_review.config,
    {"port": "Los Angeles"},
)
forked = travel_app.invoke(None, fork_config)

print("original =", original["verdict"])
print("replayed =", replayed["verdict"])
print("forked   =", forked["verdict"])

assert replayed["verdict"] == original["verdict"]
assert forked["verdict"] == "reviewed:Los Angeles:lithium-battery"
assert original["verdict"] == "reviewed:Shanghai:lithium-battery"
print("04 checkpoint replay + fork ok")

<!-- codex:checklist -->
---

## 练习任务 Checklist

完成后逐项勾选：

- [ ] 证明相同 `thread_id` 可以续聊，不同 thread 不携带旧消息。
- [ ] 使用 `get_state()` 找到当前 checkpoint ID。
- [ ] 证明同一 user 跨 thread 保留偏好，不同 user 相互隔离。
- [ ] 使用 `get_state_history()` 列出历史 checkpoint。
- [ ] 从历史 checkpoint 完成一次 replay。
- [ ] 使用 `update_state()` 创建 fork，并证明原分支未被覆盖。

**交付证据：**thread/user 对照、历史列表、original/replayed/forked 三组结果。